In [1]:

# ============================================================
# CELL 1: Imports & Configuration
# ============================================================

import sqlite3
import pandas as pd
import numpy as np
import warnings
import time
import math
from datetime import datetime

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('future.no_silent_downcasting', True)

DB_PATH = r'C:\Users\Sarthak\Documents\ML\fighter-beta\mma_fighters.db'
print("✓ Imports complete")

C:\Users\Sarthak\miniconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


✓ Imports complete


In [5]:

# ============================================================
# CELL 2: Database Connection
# ============================================================

conn = sqlite3.connect(DB_PATH)

# Verify
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"✓ Connected. Tables: {len(tables)}")

fights_count = pd.read_sql("SELECT COUNT(*) as c FROM fights_v2", conn)['c'].iloc[0]
print(f"✓ Total fights in DB: {fights_count}")

✓ Connected. Tables: 10
✓ Total fights in DB: 11127


In [6]:

# ============================================================
# CELL 3: Helper Functions
# ============================================================

def parse_reach(r):
    if pd.isna(r) or r == '--': return None
    try: return float(r.replace('"', ''))
    except: return None

def parse_height(h):
    if pd.isna(h) or h == '--': return None
    try:
        parts = h.replace('"', '').split("'")
        return int(parts[0]) * 12 + int(parts[1])
    except: return None

def parse_age(dob, fight_date):
    if pd.isna(dob) or pd.isna(fight_date): return None
    try:
        birth = datetime.strptime(dob, "%b %d, %Y")
        fight = datetime.strptime(fight_date, "%Y-%m-%d")
        return (fight - birth).days / 365.25
    except: return None

def parse_fight_duration(ending_round, ending_time):
    try:
        mins, secs = ending_time.split(':')
        final_round_minutes = int(mins) + int(secs) / 60
        return ((ending_round - 1) * 5) + final_round_minutes
    except: return 15.0

# Time decay: 1.5 year half-life
LAM = math.log(2) / (1.5 * 365)

def time_decay_weights(dates, as_of_date, lam=LAM):
    as_of = datetime.strptime(as_of_date, "%Y-%m-%d")
    weights = []
    for d in dates:
        fight_dt = datetime.strptime(d, "%Y-%m-%d")
        days_ago = (as_of - fight_dt).days
        w = np.exp(-lam * days_ago)
        weights.append(w)
    weights = np.array(weights)
    return weights / weights.sum() if weights.sum() > 0 else weights

print(f"✓ Lambda (1.5yr half-life): {LAM:.6f}")
print("✓ Helper functions ready")

✓ Lambda (1.5yr half-life): 0.001266
✓ Helper functions ready


In [7]:
# ============================================================
# CELL 4: Pre-Cache Fight Data (2016+, 3+ fights filter)
# ============================================================

print("Fetching all fight stats (2016+ only)...")
start = time.time()

# Fetch all fight-level stats
all_fight_stats = pd.read_sql("""
    SELECT 
        f.fight_id,
        f.event_date,
        f.weight_class,
        f.ending_round,
        f.ending_time,
        f.method,
        ff.fighter_id,
        SUM(frs.sig_str_landed) as sig_str_landed,
        SUM(frs.sig_str_attempted) as sig_str_attempted,
        SUM(frs.td_landed) as td_landed,
        SUM(frs.td_attempted) as td_attempted,
        SUM(frs.sub_attempts) as sub_attempts,
        SUM(frs.ctrl_seconds) as ctrl_seconds,
        SUM(frs.knockdowns) as knockdowns,
        COUNT(DISTINCT fr.round_number) as total_rounds
    FROM fight_round_stats_v2 frs
    JOIN fight_rounds_v2 fr ON frs.round_id = fr.round_id
    JOIN fights_v2 f ON fr.fight_id = f.fight_id
    JOIN fight_fighters_v2 ff ON f.fight_id = ff.fight_id AND frs.fighter_id = ff.fighter_id
    WHERE f.event_date >= '2016-01-01'
    GROUP BY f.fight_id, ff.fighter_id
""", conn)

# Calculate per-minute stats
all_fight_stats['actual_minutes'] = all_fight_stats.apply(
    lambda r: parse_fight_duration(r['ending_round'], r['ending_time']), axis=1
)

all_fight_stats['slpm'] = all_fight_stats['sig_str_landed'] / all_fight_stats['actual_minutes']
all_fight_stats['str_acc'] = all_fight_stats['sig_str_landed'] / all_fight_stats['sig_str_attempted']
all_fight_stats['td_acc'] = all_fight_stats['td_landed'] / all_fight_stats['td_attempted']
all_fight_stats['td_avg'] = all_fight_stats['td_landed'] / (all_fight_stats['actual_minutes'] / 15)
all_fight_stats['sub_avg'] = all_fight_stats['sub_attempts'] / (all_fight_stats['actual_minutes'] / 15)
all_fight_stats['ctrl_time_per_min'] = all_fight_stats['ctrl_seconds'] / all_fight_stats['actual_minutes'] / 60
all_fight_stats['kd_per_min'] = all_fight_stats['knockdowns'] / all_fight_stats['actual_minutes']

all_fight_stats = all_fight_stats.replace([np.inf, -np.inf], np.nan).fillna(0)
all_fight_stats = all_fight_stats.sort_values('event_date').reset_index(drop=True)

# Build fighter history dict
fighter_fights_dict = {
    fid: group.sort_values('event_date').reset_index(drop=True)
    for fid, group in all_fight_stats.groupby('fighter_id')
}

# Opponent lookup
all_opponents = pd.read_sql("""
    SELECT ff1.fight_id, ff1.fighter_id, ff2.fighter_id as opponent_id
    FROM fight_fighters_v2 ff1
    JOIN fight_fighters_v2 ff2 ON ff1.fight_id = ff2.fight_id 
        AND ff1.fighter_id != ff2.fighter_id
""", conn)

opponents_dict = {
    (row['fight_id'], row['fighter_id']): row['opponent_id']
    for _, row in all_opponents.iterrows()
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✓ Fighters cached: {len(fighter_fights_dict)}")
print(f"✓ Fight records: {len(all_fight_stats)}")
print(f"✓ Opponent lookups: {len(opponents_dict)}")

Fetching all fight stats (2016+ only)...
Done in 2.6s
✓ Fighters cached: 2135
✓ Fight records: 11107
✓ Opponent lookups: 22254


In [8]:
# ============================================================
# CELL 5: Generate Base Fights (2016+, Both Fighters 3+ UFC Fights)
# ============================================================

def generate_base_fights_filtered(start_date='2016-01-01', end_date='2022-12-31'):
    """
    Generate base fight dataset with CRITICAL filter:
    - Both fighters must have 3+ previous UFC fights
    - This matches working model's data filtering
    """
    
    fights = pd.read_sql(f"""
        SELECT 
            f.fight_id,
            f.event_date,
            f.event_name,
            f.weight_class,
            f.method,
            f.ending_round,
            f.ending_time,
            ff1.fighter_id as fighter_1_id,
            fv1.name as fighter_1_name,
            ff2.fighter_id as fighter_2_id,
            fv2.name as fighter_2_name,
            CASE WHEN ff1.result = 'win' THEN 1 ELSE 0 END as winner
        FROM fights_v2 f
        JOIN fight_fighters_v2 ff1 ON f.fight_id = ff1.fight_id AND ff1.corner = 'fighter_1'
        JOIN fight_fighters_v2 ff2 ON f.fight_id = ff2.fight_id AND ff2.corner = 'fighter_2'
        JOIN fighters_v2 fv1 ON ff1.fighter_id = fv1.fighter_id
        JOIN fighters_v2 fv2 ON ff2.fighter_id = fv2.fighter_id
        WHERE f.event_date >= '{start_date}'
        AND f.event_date <= '{end_date}'
        ORDER BY f.event_date
    """, conn)
    
    # Filter: Both fighters must have 3+ previous UFC fights
    valid_fights = []
    
    for idx, fight in fights.iterrows():
        f1_id = fight['fighter_1_id']
        f2_id = fight['fighter_2_id']
        fight_date = fight['event_date']
        
        # Count previous fights for each fighter
        if f1_id in fighter_fights_dict:
            f1_prev = fighter_fights_dict[f1_id]
            f1_count = len(f1_prev[f1_prev['event_date'] < fight_date])
        else:
            f1_count = 0
        
        if f2_id in fighter_fights_dict:
            f2_prev = fighter_fights_dict[f2_id]
            f2_count = len(f2_prev[f2_prev['event_date'] < fight_date])
        else:
            f2_count = 0
        
        # Only include if BOTH have 3+ previous fights
        if f1_count >= 3 and f2_count >= 3:
            valid_fights.append(fight)
    
    filtered = pd.DataFrame(valid_fights)
    
    print(f"Original fights (2016-{end_date[:4]}): {len(fights)}")
    print(f"After 3+ fight filter: {len(filtered)}")
    print(f"Dropped: {len(fights) - len(filtered)} ({(len(fights)-len(filtered))/len(fights)*100:.1f}%)")
    
    return filtered


# Generate filtered dataset
base_fights = generate_base_fights_filtered(start_date='2016-01-01', end_date='2022-12-31')
print(f"\n✓ Base fights shape: {base_fights.shape}")
print(f"✓ Date range: {base_fights['event_date'].min()} to {base_fights['event_date'].max()}")
print(f"✓ Winner distribution: {base_fights['winner'].value_counts().to_dict()}")

Original fights (2016-2022): 3734
After 3+ fight filter: 1354
Dropped: 2380 (63.7%)

✓ Base fights shape: (1354, 12)
✓ Date range: 2017-01-28 to 2022-12-17
✓ Winner distribution: {0: 691, 1: 663}


In [22]:
# ============================================================
# CELL 6 (Fixed): Three-Layer with Decayed Averages & Bayesian Smoothing
# ============================================================

def bayesian_smooth(observed, n_samples, population_mean, shrinkage_factor=5):
    """Bayesian smoothing: blend observed with population mean"""
    weight = n_samples / (n_samples + shrinkage_factor)
    return weight * observed + (1 - weight) * population_mean


def calculate_three_layer_features(fighter_id, opponent_id, as_of_date, weight_class,
                                   stats=['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg',
                                          'ctrl_time_per_min', 'kd_per_min'],
                                   window=5):
    """
    Generate THREE layers with TIME-DECAYED AVERAGES (not single fight):
    1. Fighter's decayed average (last N fights, 1.5yr half-life)
    2. Opponent's baseline (decayed, MAD-based)
    3. AdjPerf z-score
    """
    
    if fighter_id not in fighter_fights_dict or opponent_id not in fighter_fights_dict:
        return None
    
    fighter_history = fighter_fights_dict[fighter_id]
    opponent_history = fighter_fights_dict[opponent_id]
    
    fighter_prev = fighter_history[fighter_history['event_date'] < as_of_date]
    opponent_prev = opponent_history[opponent_history['event_date'] < as_of_date]
    
    if len(fighter_prev) == 0 or len(opponent_prev) == 0:
        return None
    
    # Population means
    population_means = {
        'slpm': all_fight_stats['slpm'].median(),
        'str_acc': all_fight_stats['str_acc'].median(),
        'td_acc': all_fight_stats['td_acc'].median(),
        'td_avg': all_fight_stats['td_avg'].median(),
        'sub_avg': all_fight_stats['sub_avg'].median(),
        'ctrl_time_per_min': all_fight_stats['ctrl_time_per_min'].median(),
        'kd_per_min': all_fight_stats['kd_per_min'].median()
    }
    
    # Get recent fights with time decay
    fighter_recent = fighter_prev.tail(window)
    fighter_weights = time_decay_weights(fighter_recent['event_date'].tolist(), as_of_date, LAM)
    
    features = {}
    
    for stat in stats:
        # Layer 1: Fighter's TIME-DECAYED AVERAGE (not single fight!)
        fighter_values = fighter_recent[stat].values
        decayed_avg = np.average(fighter_values, weights=fighter_weights)
        
        # Bayesian smooth
        n_samples = len(fighter_recent)
        smoothed_stat = bayesian_smooth(decayed_avg, n_samples, population_means[stat])
        features[f'{stat}_dec_avg'] = smoothed_stat
        
        # Layer 2: Opponent's baseline (time-decayed)
        opp_allowed = []
        opp_dates = []
        
        for _, opp_fight in opponent_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
            if opp_opp_id and opp_opp_id in fighter_fights_dict:
                opp_opp_fights = fighter_fights_dict[opp_opp_id]
                opp_opp_this_fight = opp_opp_fights[opp_opp_fights['fight_id'] == opp_fight['fight_id']]
                if len(opp_opp_this_fight) > 0:
                    opp_allowed.append(opp_opp_this_fight[stat].iloc[0])
                    opp_dates.append(opp_fight['event_date'])
        
        if len(opp_allowed) >= 2:
            # Time-decay opponent's history too
            opp_weights = time_decay_weights(opp_dates, as_of_date, LAM)
            opp_median = np.average(opp_allowed, weights=opp_weights)
            
            # MAD with time decay
            abs_devs = np.abs(np.array(opp_allowed) - opp_median)
            opp_mad = np.average(abs_devs, weights=opp_weights)
            
            # Bayesian smooth
            n_opp = len(opp_allowed)
            opp_median_smoothed = bayesian_smooth(opp_median, n_opp, population_means[stat])
            opp_mad_smoothed = max(opp_mad, 0.01)
            
            features[f'{stat}_opp_dec_avg'] = opp_median_smoothed
            features[f'{stat}_opp_mad'] = opp_mad_smoothed
            
            # Layer 3: AdjPerf
            features[f'{stat}_adjperf'] = (smoothed_stat - opp_median_smoothed) / opp_mad_smoothed
            features[f'{stat}_adjperf'] = np.clip(features[f'{stat}_adjperf'], -7, 7)
        else:
            features[f'{stat}_opp_dec_avg'] = population_means[stat]
            features[f'{stat}_opp_mad'] = 1.0
            features[f'{stat}_adjperf'] = 0
    
    return features


print("✓ Three-layer with TIME-DECAYED AVERAGES ready")
print("  Layer 1: Fighter's decayed avg (last 5 fights, 1.5yr half-life)")
print("  Layer 2: Opponent's decayed baseline (MAD)")
print("  Layer 3: AdjPerf z-score")

✓ Three-layer with TIME-DECAYED AVERAGES ready
  Layer 1: Fighter's decayed avg (last 5 fights, 1.5yr half-life)
  Layer 2: Opponent's decayed baseline (MAD)
  Layer 3: AdjPerf z-score


In [23]:
# ============================================================
# CELL 7: Generate Features for All Fights
# ============================================================

def generate_features(base_df):
    """Generate all three layers for both fighters + differences"""
    
    rows = []
    
    for idx, fight in base_df.iterrows():
        if idx % 200 == 0:
            print(f"  Processing {idx}/{len(base_df)}...")
        
        row = {'fight_id': fight['fight_id']}
        
        # Fighter 1 features
        f1_features = calculate_three_layer_features(
            fight['fighter_1_id'], 
            fight['fighter_2_id'],
            fight['event_date'],
            fight['weight_class']
        )
        
        # Fighter 2 features  
        f2_features = calculate_three_layer_features(
            fight['fighter_2_id'],
            fight['fighter_1_id'], 
            fight['event_date'],
            fight['weight_class']
        )
        
        if f1_features is None or f2_features is None:
            continue
        
        # Add all features
        for key, val in f1_features.items():
            row[f'f1_{key}'] = val
        
        for key, val in f2_features.items():
            row[f'f2_{key}'] = val
        
        # Difference features (what model learns from)
        for key in f1_features.keys():
            row[f'diff_{key}'] = row[f'f1_{key}'] - row[f'f2_{key}']
        
        rows.append(row)
    
    features_df = pd.DataFrame(rows)
    return base_df.merge(features_df, on='fight_id', how='inner')


print("Generating three-layer features...")
start = time.time()
fight_features = generate_features(base_fights)
print(f"Done in {time.time()-start:.1f}s")
print(f"✓ Shape: {fight_features.shape}")
print(f"✓ Features per fighter: {len([c for c in fight_features.columns if c.startswith('f1_')])}")

Generating three-layer features...
  Processing 1200/1354...
  Processing 2000/1354...
  Processing 2200/1354...
  Processing 2800/1354...
Done in 47.8s
✓ Shape: (1354, 96)
✓ Features per fighter: 28


In [24]:
# ============================================================
# CELL 8: Physical Features & Experience
# ============================================================

def add_physical_and_experience(df, conn):
    """Add physical attributes and UFC experience"""
    
    all_ids = set(df['fighter_1_id'].unique()) | set(df['fighter_2_id'].unique())
    
    fighters = pd.read_sql(f"""
        SELECT fighter_id, reach, height, dob
        FROM fighters_v2
        WHERE fighter_id IN ({','.join(f"'{fid}'" for fid in all_ids)})
    """, conn)
    
    fighters['reach_in'] = fighters['reach'].apply(parse_reach)
    fighters['height_in'] = fighters['height'].apply(parse_height)
    
    # Merge fighter 1
    df = df.merge(
        fighters[['fighter_id', 'reach_in', 'height_in', 'dob']],
        left_on='fighter_1_id', right_on='fighter_id', how='left'
    ).drop('fighter_id', axis=1).rename(columns={
        'reach_in': 'f1_reach', 'height_in': 'f1_height', 'dob': 'f1_dob'
    })
    
    # Merge fighter 2
    df = df.merge(
        fighters[['fighter_id', 'reach_in', 'height_in', 'dob']],
        left_on='fighter_2_id', right_on='fighter_id', how='left'
    ).drop('fighter_id', axis=1).rename(columns={
        'reach_in': 'f2_reach', 'height_in': 'f2_height', 'dob': 'f2_dob'
    })
    
    # Calculate age
    df['f1_age'] = df.apply(lambda r: parse_age(r['f1_dob'], r['event_date']), axis=1)
    df['f2_age'] = df.apply(lambda r: parse_age(r['f2_dob'], r['event_date']), axis=1)
    
    # Differences and ratios
    df['age_diff'] = df['f1_age'] - df['f2_age']
    df['reach_diff'] = df['f1_reach'] - df['f2_reach']
    df['height_diff'] = df['f1_height'] - df['f2_height']
    
    df['age_ratio'] = df['f1_age'] / df['f2_age']
    df['reach_ratio'] = df['f1_reach'] / df['f2_reach']
    df['height_ratio'] = df['f1_height'] / df['f2_height']
    
    # UFC Age (time in promotion)
    for prefix, fid_col in [('f1', 'fighter_1_id'), ('f2', 'fighter_2_id')]:
        ufc_ages = []
        for _, fight in df.iterrows():
            fid = fight[fid_col]
            if fid in fighter_fights_dict:
                first_fight = fighter_fights_dict[fid].iloc[0]['event_date']
                days = (datetime.strptime(fight['event_date'], "%Y-%m-%d") - 
                       datetime.strptime(first_fight, "%Y-%m-%d")).days
                ufc_ages.append(days / 365.25)
            else:
                ufc_ages.append(0)
        df[f'{prefix}_ufc_age'] = ufc_ages
    
    df['diff_ufc_age'] = df['f1_ufc_age'] - df['f2_ufc_age']
    
    # Drop raw columns
    df = df.drop(['f1_dob', 'f2_dob'], axis=1)
    
    return df


fight_features = add_physical_and_experience(fight_features, conn)
print(f"✓ Shape with physical features: {fight_features.shape}")

✓ Shape with physical features: (1354, 111)


In [25]:
# ============================================================
# CELL 9: Train Model
# ============================================================

# Time-based split
train_data = fight_features[fight_features['event_date'] < '2021-01-01']
val_data   = fight_features[fight_features['event_date'] >= '2021-01-01']

print(f"Train: {len(train_data)} fights (2017-2020)")
print(f"Val: {len(val_data)} fights (2021-2022)")

# Keep only difference features + physical
drop_cols = ['fight_id', 'event_date', 'event_name', 'weight_class', 'method',
             'ending_round', 'ending_time', 'fighter_1_id', 'fighter_1_name',
             'fighter_2_id', 'fighter_2_name', 'winner',
             'f1_reach', 'f2_reach', 'f1_height', 'f2_height', 
             'f1_age', 'f2_age', 'f1_ufc_age', 'f2_ufc_age']

# Remove individual fighter features, keep only diffs + physical ratios
keep_cols = [c for c in fight_features.columns 
             if c.startswith('diff_') 
             or c in ['age_diff', 'reach_diff', 'height_diff',
                      'age_ratio', 'reach_ratio', 'height_ratio']]

X_train = train_data[keep_cols].fillna(0)
y_train = train_data['winner']
X_val   = val_data[keep_cols].fillna(0)
y_val   = val_data['winner']

print(f"Features: {len(keep_cols)}")

model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    min_child_weight=8,
    subsample=0.7,
    colsample_bytree=0.7,
    gamma=1.0,
    random_state=42
)

model.fit(X_train, y_train)

print(f"\n{'='*50}")
print(f"v2 (old approach): 59.5% validation")
print(f"{'='*50}")
print(f"v3 (corrected methodology):")
print(f"Training:   {model.score(X_train, y_train):.1%}")
print(f"Validation: {model.score(X_val, y_val):.1%}")
print(f"Gap:        {(model.score(X_train, y_train) - model.score(X_val, y_val))*100:.1f}%")

Train: 744 fights (2017-2020)
Val: 610 fights (2021-2022)
Features: 35

v2 (old approach): 59.5% validation
v3 (corrected methodology):
Training:   89.4%
Validation: 59.5%
Gap:        29.9%


In [26]:
# Test with 2+ fight filter instead of 3+
def count_fights_by_filter(min_fights):
    valid = 0
    for _, fight in base_fights.iterrows():
        f1_id = fight['fighter_1_id']
        f2_id = fight['fighter_2_id']
        fight_date = fight['event_date']
        
        f1_count = len(fighter_fights_dict.get(f1_id, pd.DataFrame())[
            fighter_fights_dict.get(f1_id, pd.DataFrame())['event_date'] < fight_date
        ]) if f1_id in fighter_fights_dict else 0
        
        f2_count = len(fighter_fights_dict.get(f2_id, pd.DataFrame())[
            fighter_fights_dict.get(f2_id, pd.DataFrame())['event_date'] < fight_date
        ]) if f2_id in fighter_fights_dict else 0
        
        if f1_count >= min_fights and f2_count >= min_fights:
            valid += 1
    return valid

print("Fights available by minimum previous fight requirement:")
print(f"1+ fights: {count_fights_by_filter(1)}")
print(f"2+ fights: {count_fights_by_filter(2)}")
print(f"3+ fights: {count_fights_by_filter(3)} (current)")

Fights available by minimum previous fight requirement:
1+ fights: 1354
2+ fights: 1354
3+ fights: 1354 (current)


In [27]:
# Check how many fights you'll have total and in each split
check = pd.read_sql("""
    SELECT 
        COUNT(*) as total_fights,
        SUM(CASE WHEN event_date < '2024-01-01' THEN 1 ELSE 0 END) as train_fights,
        SUM(CASE WHEN event_date >= '2024-01-01' THEN 1 ELSE 0 END) as val_fights,
        MIN(event_date) as earliest,
        MAX(event_date) as latest
    FROM fights_v2
    WHERE method NOT IN ('dq', 'overturned')
""", conn)
print(check)

   total_fights  train_fights  val_fights    earliest      latest
0         11127          9896        1231  1993-11-12  2026-02-07


In [28]:
methods = pd.read_sql("""
    SELECT method, COUNT(*) as count 
    FROM fights_v2 
    GROUP BY method 
    ORDER BY count DESC
""", conn)
print(methods)

       method  count
0      KO/TKO   3753
1       U-DEC   3750
2         SUB   2351
3       S-DEC    993
4       M-DEC    111
5  Overturned     70
6         CNC     47
7          DQ     35
8       Other     14
9    Decision      3
